In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import ast, math, re
import pandas as pd
from pathlib import Path

In [ ]:
allowed = {k: v for k, v in vars(math).items() if not k.startswith("__")}
allowed.update({"np": np})

def safe_eval(expr: str):
    expr_ast = ast.parse(expr, mode="eval")         
    for node in ast.walk(expr_ast):                  # check
        if isinstance(node, ast.Name) and node.id not in allowed:
            raise ValueError(f"Wrong name: {node.id}")
    compiled = compile(expr_ast, "<string>", "eval") 
    return eval(compiled, {"__builtins__": {}}, allowed)

def load_params(fname="05282025_test.txt"):
    pat = re.compile(r"^\s*([^#=\s]+)\s*=\s*(.+?)\s*$")
    P = {}
    with open(fname, encoding="utf-8") as f:
        for line in f:
            if not line.strip() or line.lstrip().startswith("#"):
                continue
            m = pat.match(line)
            if m:
                P[m.group(1)] = safe_eval(m.group(2))
    return P

In [ ]:
P = load_params()
# ----------Generate data----------
np.random.seed(int(P["seed_T"]))
T_amb = np.random.uniform(P["T_low"], P["T_high"], int(P["samples_T"]))
Time  = len(T_amb)

# internal activity
np.random.seed(int(P["seed_occ"]))
r1 = (P["g_appliance"] + P["g_human"]) * np.random.uniform(1, P["N_o"], Time)

q1 = np.array(
    [P["g_min"] if t < P["activity_start"] or t >= P["activity_end"] else r1[t]
     for t in range(Time)]
)
occupancy1 = np.array(
    [0 if t <= P["activity_start"] or t >= P["activity_end"]
     else r1[t] / (P["g_appliance"] + P["g_human"])
     for t in range(Time)]
)

np.random.seed(1)

tau= P["tau"]
V= P["V"]
P_I= P["P_I"] #
Ca= P["Ca"]
rho= P["rho"]     
K_amb=P["K_amb"]
COP= P["COP"] 
alpha = P["alpha"]

#Indoor temperature 
T1_min=P["T1_min"]          
T1_max=P["T1_max"]          
T_ini_new= P["T_ini"]

#air flow rate
m_min= P["m_min"]  # a
m_max= P["m_max"]  # b

#Supply air
T_sa1_min= P["T_sa1_min"]
T_sa1_max = 0.0006 * P["m_max"] + 0.317 * P["T1_max"] + 5.792
#print("T_sa1_max",T_sa1_max)

#CO2 concentration
O_gen= P["O_gen"]
O1_min = P["O1_min"]


O1_max = P["O1_max"]
O_out = P["O_out"] 
O1_ini_new = O_out
#VOC level
v_min = P["v_min"] #internal source min=0.0556; max=0.1389 ug/s*m^2

VOC_min = P["VOC_min"]
VOC_max = P["VOC_max"] #300-500 ug/m^3
VOC_out = P["VOC_out"] #5-50 ug/m^3

VOC_ini_new = VOC_out
v1 = np.full(Time, P["v_min"])
T_ini_new= P["T_ini"]

In [ ]:
pi = [] 
for t in range(Time): 
    hour = t % 24 
    if 7 <= hour < 22: 
        pi.append(0.20) # peak 
    else: 
        pi.append(0.10) # off-peak

In [ ]:
def tou_flat(Time):
    return [0.12] * Time

def tou_two_period(Time):
    pi = []
    for t in range(Time):
        hour = t % 24
        if 7 <= hour < 22:
            pi.append(0.20)
        else:
            pi.append(0.10)
    return pi

def tou_aggressive(Time):
    pi = []
    for t in range(Time):
        hour = t % 24
        if 14 <= hour < 20:
            pi.append(0.30)
        elif 7 <= hour < 22:
            pi.append(0.18)
        else:
            pi.append(0.08)
    return pi


In [ ]:
def solve_static(T_ini_new, pi, Time, tau, T_amb, q1, K_amb, rho, Ca, V,ach_inf_base,inf_multiplier=1.0):
    # 
    x_ac = {}
    P_ac = {}
    T = {}
    T_set = {}
    s_T = {}

    M_big = 10000
    m = gp.Model("static_AC_new")

    for t in range(Time):
        x_ac[t] = m.addVar(vtype=GRB.BINARY, name=f"x_ac_{t}")
        P_ac[t] = m.addVar(lb=0, vtype=GRB.CONTINUOUS, name=f"P_ac_{t}")
        T[t] = m.addVar(lb=0, vtype=GRB.CONTINUOUS, name=f"T_{t}")
        T_set[t] = m.addVar(lb=0, vtype=GRB.CONTINUOUS, name=f"Tset_{t}")
        s_T[t] = m.addVar(lb=0, vtype=GRB.CONTINUOUS, name=f"sT_{t}")

    # 
    m.addConstr(T[0] == T_ini_new, name="T_init")

    # P_t^{AC} = 1000 * x_t^{AC}
    for t in range(Time):
        m.addConstr(P_ac[t] == 1000 * x_ac[t], name=f"link_P_x_{t}")


    ach_inf_base_ts = np.full(Time, ach_inf_base) if np.isscalar(ach_inf_base) else np.array(ach_inf_base, dtype=float)
    ach_inf_ts = inf_multiplier * ach_inf_base_ts               # ACH (1/h)
    lam_inf_ts = ach_inf_ts / 3600.0                            # 1/s

    for t in range(Time - 1):
        m.addConstr(
            T[t+1] == T[t]
            + ((-0.7) * P_ac[t]  + q1[t]) * tau / (rho * Ca * V)
            + (lam_inf_ts[t] * (T_amb[t] - T[t])) * tau,   # 
            name=f"thermal_dyn_{t}"
        )
    # 

    # hysteresis / setpoint
    for t in range(Time):
        m.addConstr(T[t] - T_set[t] - 0.5 <= M_big * x_ac[t], name=f"hys_up_{t}")
        m.addConstr(T_set[t] - 0.5 - T[t] <= M_big * (1 - x_ac[t]), name=f"hys_down_{t}")

    # setpoint
    for t in range(Time):
        m.addConstr(17 <= T_set[t], name=f"comfort_low_{t}")
        m.addConstr(T_set[t] <= 26, name=f"comfort_high_{t}")

    # energy cost
    energy_cost = gp.quicksum(P_ac[t] * pi[t] * tau / 3600 for t in range(Time))
    m.setObjective(energy_cost, GRB.MINIMIZE)

    m.setParam("MIPGap", 0.041)
    m.setParam("TimeLimit", 200)

    m.optimize()

    if m.status not in [GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.SUBOPTIMAL]:
        return None

    sol = {
        "status": m.status,
        "obj": float(m.ObjVal) if m.SolCount > 0 else None,
        "x_ac": np.array([x_ac[t].X for t in range(Time)]) if m.SolCount > 0 else None,
        "P_ac": np.array([P_ac[t].X for t in range(Time)]) if m.SolCount > 0 else None,
        "T": np.array([T[t].X for t in range(Time)]) if m.SolCount > 0 else None,
        "T_set": np.array([T_set[t].X for t in range(Time)]) if m.SolCount > 0 else None,
    }
    return sol

In [ ]:
def recompute_metrics(sol, pi, tau):
    """
    sol: solve_static 返回的 dict
    pi: list/np.array, length=Time
    tau: 秒 (e.g., 3600 or 900)
    """
    P_ac = sol["P_ac"]          # W
    x_ac = sol["x_ac"]
    T = sol["T"]
    T_set = sol["T_set"]
    pi = np.array(pi, dtype=float)

    dt_h = tau / 3600.0  # hours

    # (kWh) = P(W) * dt(h) / 1000
    energy_kWh_t = P_ac * dt_h / 1000.0
    total_energy_kWh = float(np.sum(energy_kWh_t))


    # 
    cost_like_model_t = P_ac * pi * dt_h                 # 
    total_cost_like_model = float(np.sum(cost_like_model_t))

    cost_kWh_based_t = (P_ac / 1000.0) * pi * dt_h       # 
    total_cost_kWh_based = float(np.sum(cost_kWh_based_t))

    # 
    temp_error = T - T_set
    max_abs_temp_error = float(np.max(np.abs(temp_error)))

    metrics = {
        "total_energy_kWh": total_energy_kWh,
        "total_cost_like_model": total_cost_like_model,
        "total_cost_kWh_based": total_cost_kWh_based,
        "energy_kWh_t": energy_kWh_t,
        "cost_like_model_t": cost_like_model_t,
        "cost_kWh_based_t": cost_kWh_based_t,
        "max_abs_T_minus_Tset": max_abs_temp_error,
        "on_fraction": float(np.mean(x_ac > 0.5)),
    }
    return metrics

In [ ]:
all_results = {}

tou_map = {
    "flat": tou_flat,
    "two_period": tou_two_period,
    "aggressive": tou_aggressive
}

for name, tou_func in tou_map.items():
    pi = tou_func(Time)

    sol = solve_static(
        T_ini_new=T_ini_new,
        pi=pi,
        Time=Time,
        tau=tau,
        T_amb=T_amb,
        q1=q1,
        K_amb=K_amb,
        rho=rho,
        Ca=Ca,
        V=V
    )

    if sol is None or sol["P_ac"] is None:
        print(f"{name}: no solution")
        continue

    metrics = recompute_metrics(sol, pi, tau)

    all_results[name] = {
        "sol": sol,
        "metrics": metrics
    }

    print(
        name,
        "obj(model)=", sol["obj"],
        "total_energy_kWh=", metrics["total_energy_kWh"],
        "total_cost_like_model=", metrics["total_cost_like_model"],
        "total_cost_kWh_based=", metrics["total_cost_kWh_based"]
    )

In [ ]:
import numpy as np

def simulate_iaq(
    C0,
    Cout,
    G_t,
    V,
    tau,
    ach_inf_base,     # base infiltration in ACH (1/h), length=Time or scalar
    inf_multiplier=1.0,
    ach_mech=None,    # optional mechanical outdoor air ACH (1/h), length=Time or scalar
    k_decay_per_h=0.0 # optional decay/removal (1/h)
):
    """
    Return C(t) length=Time, using explicit Euler.

    Units:
    - CO2: use ppm for C0/Cout, and G_t should be "ppm*m3/s"? (see note below)
    - VOC: use mg/m3, and G_t should be mg/s

    For practicality: we implement in concentration units with:
      dC/dt = (G_t / V) + (λ_total)*(Cout - C) - k*C
    where λ_total = (ACH_inf*mult + ACH_mech)/3600 (1/s)
          k = k_decay_per_h/3600 (1/s)
    """
    Time = len(G_t)
    C = np.zeros(Time)
    C[0] = C0

    # make time series
    ach_inf_base = np.full(Time, ach_inf_base) if np.isscalar(ach_inf_base) else np.array(ach_inf_base, dtype=float)
    ach_mech = np.zeros(Time) if ach_mech is None else (np.full(Time, ach_mech) if np.isscalar(ach_mech) else np.array(ach_mech, dtype=float))

    lam = (inf_multiplier * ach_inf_base + ach_mech) / 3600.0  # 1/s
    k = k_decay_per_h / 3600.0                                 # 1/s

    for t in range(Time - 1):
        dCdt = (G_t[t] / V) + lam[t] * (Cout - C[t]) - k * C[t]
        C[t+1] = C[t] + tau * dCdt

    return C

In [ ]:

Time = Time
tau = tau
V = V  # m3
occupancy = occupancy1  # length=Time


Cout_co2 = 420.0   # ppm
Cout_voc = 0.30    # mg/m3 


C0_co2 = Cout_co2
C0_voc = Cout_voc


ach_inf_base = 0.3 


G_co2 = 5.0 * occupancy                 # ppm*m3/s 
G_voc = 0.002 * occupancy               # mg/s 

G_co2 = 5.0 * occupancy1    # ppm*m3/s 
Cout_co2 = 420.0
C0_co2 = Cout_co2
G_voc = 0.002 * occupancy1  # mg/s 
Cout_voc = 0.2
C0_voc = Cout_voc
G_activity = np.zeros(Time)
for t in range(Time):
    hour = t % 24
    if 18 <= hour < 20:
        G_activity[t] += 0.01
    if 10 <= hour < 11:
        G_activity[t] += 0.005

G_voc = (
    0.0005 * np.ones(Time)
    + 0.0015 * np.ones(Time)
    + 0.006 * occupancy1
    + G_activity
)
multipliers = [1.0, 2.0, 4.0]           # higher infiltration levels

co2_profiles = {}
voc_profiles = {}

for m in multipliers:
    co2_profiles[m] = simulate_iaq(C0_co2, Cout_co2, G_co2, V, tau, ach_inf_base, inf_multiplier=m, ach_mech=0.0)
    voc_profiles[m] = simulate_iaq(C0_voc, Cout_voc, G_voc, V, tau, ach_inf_base, inf_multiplier=m, k_decay_per_h=0.2,ach_mech=0.0)

# 
for m in multipliers:
    print(f"mult={m}: CO2 peak={co2_profiles[m].max():.1f} ppm, VOC peak={voc_profiles[m].max():.4f} mg/m3")

In [ ]:

def kpi_iaq(C, tau, threshold=None):
    # C: length=Time
    peak = float(np.max(C))
    avg  = float(np.mean(C))
    exposure = float(np.sum(C) * (tau/3600.0))  
    out = {"peak": peak, "avg": avg, "exposure": exposure}
    if threshold is not None:
        out["hours_above"] = float(np.sum(C > threshold) * (tau/3600.0))
    return out

def kpi_energy(sol, pi, tau):
    P = sol["P_ac"]  # W
    dt_h = tau/3600.0
    energy_kWh = float(np.sum(P * dt_h / 1000.0))
    cost_kWh_based = float(np.sum((P/1000.0) * np.array(pi) * dt_h)) 
    return {"energy_kWh": energy_kWh, "cost": cost_kWh_based}
ach_inf_base = 0.3                
inf_multipliers = [1.0, 1.5, 2.0,2.5,3,3.5, 4.0]   # higher infiltration levels

tou_map = {
    "flat": tou_flat,
    "two_period": tou_two_period,
    "aggressive": tou_aggressive
}

rows = []

for tou_name, tou_func in tou_map.items():
    pi = tou_func(Time)

    for m_inf in inf_multipliers:
        sol = solve_static(
            T_ini_new=T_ini_new,
            pi=pi,
            Time=Time,
            tau=tau,
            T_amb=T_amb,
            q1=q1,
            K_amb=K_amb,
            rho=rho,
            Ca=Ca,
            V=V,
            ach_inf_base=ach_inf_base,
            inf_multiplier=m_inf
        )
        if sol is None or sol["P_ac"] is None:
            continue

        # energy/cost
        e = kpi_energy(sol, pi, tau)

        # IAQ
        co2 = simulate_iaq(C0=Cout_co2, Cout=Cout_co2, G_t=G_co2, V=V, tau=tau,
                           ach_inf_base=ach_inf_base, inf_multiplier=m_inf, ach_mech=None, k_decay_per_h=0.0)
        voc = simulate_iaq(C0=Cout_voc, Cout=Cout_voc, G_t=G_voc, V=V, tau=tau,
                           ach_inf_base=ach_inf_base, inf_multiplier=m_inf, ach_mech=None, k_decay_per_h=0.2)

        co2_k = kpi_iaq(co2, tau, threshold=1000.0)   # 
        voc_k = kpi_iaq(voc, tau, threshold=None)     # 

        rows.append({
            "TOU": tou_name,
            "inf_multiplier": m_inf,
            "energy_kWh": e["energy_kWh"],
            "cost": e["cost"],
            "CO2_peak": co2_k["peak"],
            "CO2_hours_above_1000": co2_k.get("hours_above", np.nan),
            "VOC_peak": voc_k["peak"],
            "VOC_exposure": voc_k["exposure"],
        })

import pandas as pd
df_tradeoff = pd.DataFrame(rows)
df_tradeoff

In [ ]:
import matplotlib.pyplot as plt

# x=energy, y=CO2_peak
plt.figure()
for tou_name in df_tradeoff["TOU"].unique():
    d = df_tradeoff[df_tradeoff["TOU"] == tou_name].sort_values("inf_multiplier")
    plt.plot(d["energy_kWh"], d["CO2_peak"], marker="o", label=tou_name)
plt.xlabel("Total HVAC energy (kWh)")
plt.ylabel("CO2 peak (ppm)")
plt.legend()
plt.show()

# x=energy, y=VOC_peak
plt.figure()
for tou_name in df_tradeoff["TOU"].unique():
    d = df_tradeoff[df_tradeoff["TOU"] == tou_name].sort_values("inf_multiplier")
    plt.plot(d["energy_kWh"], d["VOC_peak"], marker="o", label=tou_name)
plt.xlabel("Total Window AC energy (kWh)")
plt.ylabel("VOC peak (mg/m3)")
plt.legend()
plt.show()

import matplotlib.pyplot as plt

plt.figure()

for tou_name in df_tradeoff["TOU"].unique():
    d = df_tradeoff[df_tradeoff["TOU"] == tou_name].sort_values("inf_multiplier")
    plt.plot(d["energy_kWh"], d["CO2_peak"], marker="o", label=tou_name)

    # 
    for _, r in d.iterrows():
        plt.annotate(f"inf={r['inf_multiplier']:.0f}",
                     (r["energy_kWh"], r["CO2_peak"]),
                     textcoords="offset points", xytext=(6,6))

plt.xlabel("Total HVAC energy (kWh)")
plt.ylabel("CO2 peak (ppm)")
plt.legend()
plt.show()
import matplotlib.pyplot as plt

def plot_tradeoff(df, x_col="energy_kWh", y_col="CO2_peak",
                  x_label="Total HVAC energy (kWh)",
                  y_label="CO$_2$ peak (ppm)",
                  title="IAQ–Energy trade-off under different TOU tariffs",
                  annotate_points=True,
                  add_direction_arrow=True,
                  save_path=None):
    """
    df columns expected:
      - TOU (category)
      - inf_multiplier (numeric)
      - x_col, y_col
    """
    plt.figure()
    for tou_name in df["TOU"].unique():
        d = df[df["TOU"] == tou_name].sort_values("inf_multiplier")
        plt.plot(d[x_col], d[y_col], marker="o", label=tou_name)

        if annotate_points:
            for _, r in d.iterrows():
                plt.annotate(
                    f"{r['inf_multiplier']:.0f}×",
                    (r[x_col], r[y_col]),
                    textcoords="offset points",
                    xytext=(6, 6),
                    fontsize=9
                )

    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.title(title)
    plt.legend()

    # 
    if add_direction_arrow:
        # 
        tou0 = df["TOU"].unique()[0]
        d0 = df[df["TOU"] == tou0].sort_values("inf_multiplier")
        if len(d0) >= 2:
            x0, y0 = d0.iloc[0][x_col], d0.iloc[0][y_col]
            x1, y1 = d0.iloc[-1][x_col], d0.iloc[-1][y_col]
            plt.annotate(
                "infiltration increases",
                xy=(x1, y1),
                xytext=(x0, y0),
                arrowprops=dict(arrowstyle="->"),
                fontsize=9
            )

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()


# trade-off（Energy vs CO2 peak）
plot_tradeoff(
    df_tradeoff,
    x_col="energy_kWh",
    y_col="CO2_peak",
    x_label="Total HVAC energy (kWh)",
    y_label="CO$_2$ peak (ppm)",
    title="IAQ–Energy trade-off (infiltration sweep: 1×, 2×, 4×)",
    annotate_points=True,
    add_direction_arrow=True,
    save_path=None  # e.g., "tradeoff_energy_co2.png"
)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
import numpy as np

marker_map = {
    "flat": "o",
    "two_period": "s",
    "aggressive": "^"
}

fig, ax = plt.subplots(figsize=(7, 5))

# infiltration multiplier
norm = mpl.colors.Normalize(
    vmin=df_tradeoff["inf_multiplier"].min(),
    vmax=df_tradeoff["inf_multiplier"].max()
)
cmap = plt.cm.viridis

for tou_name in df_tradeoff["TOU"].unique():
    d = df_tradeoff[df_tradeoff["TOU"] == tou_name]
    sc = ax.scatter(
        d["energy_kWh"],
        d["CO2_peak"],
        c=d["inf_multiplier"],
        cmap=cmap,
        norm=norm,
        marker=marker_map[tou_name],
        s=90,
        edgecolors="black",
        linewidths=0.6,
        alpha=0.9
    )

ax.set_xlabel("Total Energy Consumption")
ax.set_ylabel("CO$_2$ peak (ppm)")
ax.set_title("IAQ–Energy trade-off under different TOU tariffs")

# colorbar ： infiltration multiplier
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("Infiltration multiplier")

#  marker TOU
tou_handles = [
    Line2D([0], [0], marker=marker_map["flat"], color="black", linestyle="None",
           markerfacecolor="lightgray", markersize=8, label="flat"),
    Line2D([0], [0], marker=marker_map["two_period"], color="black", linestyle="None",
           markerfacecolor="lightgray", markersize=8, label="two_period"),
    Line2D([0], [0], marker=marker_map["aggressive"], color="black", linestyle="None",
           markerfacecolor="lightgray", markersize=8, label="three_period")
]

ax.legend(handles=tou_handles, title="TOU", loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D

marker_map = {
    "flat": "o",
    "two_period": "s",
    "aggressive": "^"
}

fig, ax = plt.subplots(figsize=(7, 5))

# infiltration multiplier
norm = mpl.colors.Normalize(
    vmin=df_tradeoff["inf_multiplier"].min(),
    vmax=df_tradeoff["inf_multiplier"].max()
)
cmap = plt.cm.viridis

for tou_name in df_tradeoff["TOU"].unique():
    d = df_tradeoff[df_tradeoff["TOU"] == tou_name]
    sc = ax.scatter(
        d["energy_kWh"],
        d["VOC_exposure"],
        c=d["inf_multiplier"],
        cmap=cmap,
        norm=norm,
        marker=marker_map[tou_name],
        s=90,
        edgecolors="black",
        linewidths=0.6,
        alpha=0.9
    )

ax.set_xlabel("Total Energy Consumption (kWh)")
ax.set_ylabel("VOC exposure (mg·h/m$^3$)")
ax.set_title("VOC exposure–energy trade-off under different TOU tariffs")

# colorbar: infiltration multiplier
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("Infiltration multiplier")

# legend: TOU
tou_handles = [
    Line2D([0], [0], marker=marker_map["flat"], color="black", linestyle="None",
           markerfacecolor="lightgray", markersize=8, label="flat"),
    Line2D([0], [0], marker=marker_map["two_period"], color="black", linestyle="None",
           markerfacecolor="lightgray", markersize=8, label="two_period"),
    Line2D([0], [0], marker=marker_map["aggressive"], color="black", linestyle="None",
           markerfacecolor="lightgray", markersize=8, label="aggressive")
]

ax.legend(handles=tou_handles, title="TOU", loc="upper right")

plt.tight_layout()
plt.show()